## Functions

In [1]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method conzzzzt of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

In [2]:

folder="C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602"

In [3]:
#IMPORTING DATA:

directory= os.scandir(folder)
list_of_files=get_directory(folder)

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
print(newlist)


MM_final_table=[]
for item in newlist:
    #Importing data
    titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson', 'PeaksDips']
    data= pd.read_table(str(item), header=None, names=titles, sep='\t')
    data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
    data2=data2.reset_index(drop=True)
    MM_final_table.append(data2)
       

['C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160201', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160202', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160203', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160204', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160205', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20160206', 'C:/Users/Ariel/Desktop/JUPYTER/PEAKSDIPS/peaksdips_201602/MM_final_table_with_pearson_prominence_method_and_peaksdipsidentification_20

In [4]:
peaks_per_month=0
dips_per_month=0
both_per_month=0

for i in np.arange(0,len(MM_final_table)):
    if len(MM_final_table[i])!=0:
        for j in np.arange(0,len(MM_final_table[i])):
            number=MM_final_table[i]['PeaksDips'][j]
            print(number)
            if number==str(1):
                peaks_per_month=peaks_per_month+1
            if number==str(-1):  
                dips_per_month=dips_per_month+1
            if number==str(0):
                both_per_month=both_per_month+1
                
            

-1
-1
1
1
0
0
1
-1
0
0
-1
-1
1
1
-1
0
0
0
0
0
1
1
-1
0
-1
0
1
1
0
0
-1
-1
0
1
1
-1
0
1
1
0
0
0
-1


In [5]:
print("peaks_per_month")
print(peaks_per_month)
print("dips_per_month")
print(dips_per_month)
print("both_per_month")
print(both_per_month)

peaks_per_month
13
dips_per_month
12
both_per_month
18


In [6]:
'Final amount'

peaks=[0,2,0,13,22,11,36,3,0,0,2,9,14,7,18,13]
dips=[0,1,0,20,23,9,35,1,0,6,9,17,21,14,17,12]
both=[0,5,0,14,38,10,43,4,0,2,7,16,30,16,27,18]

plasmadatapermonth=[221955, 586993, 0, 1643557, 2269786, 1096338, 2384393, 144034, 50342, 1145104, 1293005, 1972944, 1868759, 2069432, 2284586, 1971580]

peaks2=[]
dips2=[]
both2=[]
for i in np.arange(0,len(peaks)):
    total=peaks[i]+dips[i]+both[i]
    if total==0:
        peaks2.append(0)
        dips2.append(0)
        both2.append(0)
    else:    
        peaks2.append(100*(peaks[i]/total))
        dips2.append(100*(dips[i]/total))
        both2.append(100*(both[i]/total))
    

print('PEAKS')
print(sum(peaks))
print('DIPS')
print(sum(dips))
print('BOTH')
print(sum(both))

PEAKS
150
DIPS
185
BOTH
230


## Amount of waves plots

In [7]:
x=np.arange(0,len(peaks))

plt.figure()
plt.plot(x,peaks,label='Peaks',color='red')
plt.plot(x,dips,label='Dips',color='blue')
plt.plot(x,both,label='Both',color='black')
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
plt.xticks(ticks=x, labels=x_labels,fontsize=13)
plt.ylabel('#', fontsize=13)
plt.ylim([0,45])
#plt
plt.legend()

In [8]:
#FIRST PLOT
plt.figure()
ax7=plt.subplot(313)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.2
plt.bar(x,both,color='red', width=barWidthnormalized, label='Normalized Data')   
plt.ylabel('# Both', fontsize=13)
#plt.xlabel('Days',fontsize=10)
#plt.legend()

#specify x-axis locations
#x_ticks = [1,31,62,90,121,151,182,212,243,274,304,335,365,396,427,456]

#specify x-axis labels
#x_labels = ['2014-Nov', '2014-Dec','2015-Jan','2015-Feb','2015-Mar','2015-Apr','2015-May','2015-Jun','2015-Jul','2015-Aug','2015-Sep','2015-Oct','2015-Nov','2015-Dec','2016-Jan','2016-Feb'] 
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
#add x-axis values to plot
#plt.xticks(ticks=x_ticks, labels=x_labels,rotation=90,fontsize=8)
plt.xticks(ticks=x, labels=x_labels,fontsize=13)


#SECOND PLOT

ax1=plt.subplot(311,sharex=ax7)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.3
plt.bar(x,peaks,color='green', width=barWidth, label='Amount of LAP data')    
plt.setp(ax1.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves',fontsize=15)
plt.ylabel('# Peaks', fontsize=13)
#plt.legend()
plt.setp(ax1.get_xticklabels(), visible=False)

#THIRD PLOT

ax2=plt.subplot(312,sharex=ax7)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.3
plt.bar(x,dips, color='blue', width=barWidth, label='Amount of Mirror Mode Waves')    
#plt.setp(ax2.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves 2015',fontsize=15)
#ax2.set_ylim([0,50]) #-10 and 10 
plt.ylabel('# Dips', fontsize=13)
#plt.legend()
plt.setp(ax2.get_xticklabels(), visible=False)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

## Normalized mm waves plots

In [9]:
peaks3=[]
dips3=[]
both3=[]

errors=[]

for i in np.arange(0,len(peaks)):
    
    if plasmadatapermonth[i]==0:
        normalizedpeaks=np.nan
        normalizeddips=np.nan
        normalizedboth=np.nan
        Error=0
        
    else:    
        normalizedpeaks=peaks[i]/plasmadatapermonth[i]
        normalizeddips=dips[i]/plasmadatapermonth[i]
        normalizedboth=both[i]/plasmadatapermonth[i]
        Error=1/plasmadatapermonth[i]
    peaks3.append(normalizedpeaks)
    dips3.append(normalizeddips)
    both3.append(normalizedboth)
    errors.append(Error)
    
errors2=errors.copy() #Not considering the values with 0
'I did it by hand, lazy'
errors2[0]=0
errors2[2]=0
errors2[8]=0



In [10]:
x=np.arange(0,len(peaks))

plt.figure()
plt.plot(x,peaks3,label='Peaks',color='red')
plt.plot(x,dips3,label='Dips',color='blue')
plt.plot(x,both3,label='Both',color='black')
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
plt.xticks(ticks=x, labels=x_labels,fontsize=13)
plt.ylabel('#', fontsize=13)
#plt.ylim([0,45])
#plt
plt.legend()


y_errormin=errors
y_errormax=errors
y_error =[y_errormin, y_errormax]

plt.errorbar(x, peaks3, yerr = y_error, fmt ='o')
plt.errorbar(x, dips3, yerr = y_error,  fmt ='o')
plt.errorbar(x, both3, yerr = y_error,  fmt ='o')

<ErrorbarContainer object of 3 artists>

# This is the important plot

In [11]:
y_errormin=errors2
y_errormax=errors
y_error =[y_errormin, y_errormax]


#FIRST PLOT
plt.figure()
ax7=plt.subplot(313)
figsize=(20,4)
barWidth=0.9
#plt.bar(x-0.3,both3,color='red', width=barWidth, label='Both')  
plt.bar(x,both3,color='black', width=barWidth, label='Both')  
#plt.bar(x+0.3,both3,color='blue', width=barWidth, label='Both')  
plt.errorbar(x, both3, yerr = y_error, fmt ='o', capsize=4)
ax7.set_ylim(0, 3.7e-5)
#plt.xlabel('Days',fontsize=10)
plt.ylabel(r'$\Omega$ '+''+'both',fontsize=10)
#plt.legend()

#x_labels = ['2014-Nov', '2014-Dec','2015-Jan','2015-Feb','2015-Mar','2015-Apr','2015-May','2015-Jun','2015-Jul','2015-Aug','2015-Sep','2015-Oct','2015-Nov','2015-Dec','2016-Jan','2016-Feb'] 
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
#add x-axis values to plot
#plt.xticks(ticks=x_ticks, labels=x_labels,rotation=90,fontsize=8)
plt.xticks(ticks=x, labels=x_labels,fontsize=13)
ax7.get_yaxis().set_label_coords(-0.04,0.5)

#SECOND PLOT

ax1=plt.subplot(311,sharex=ax7)
figsize=(20,4)
plt.bar(x,peaks3,color='black', width=barWidth, label='Peaks')  
plt.errorbar(x, peaks3, yerr = y_error, fmt ='o', capsize=4)
plt.setp(ax1.get_xticklabels(), fontsize=6)
ax1.set_ylim(0,3.7e-5)
plt.ylabel(r'$\Omega$ '+''+'peaks',fontsize=10)
#plt.title('titlee',fontsize=15)
#plt.legend()
plt.setp(ax1.get_xticklabels(), visible=False)
ax1.get_yaxis().set_label_coords(-0.04,0.5)

#THIRD PLOT

ax2=plt.subplot(312,sharex=ax7)
figsize=(20,4)
plt.bar(x,dips3, color='black', width=barWidth, label='Dips')    
plt.errorbar(x, dips3, yerr = y_error, fmt ='o', capsize=4)
ax2.set_ylim(0,3.7e-5)
plt.ylabel(r'$\Omega$ '+''+'dips',fontsize=10)
#plt.setp(ax2.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves 2015',fontsize=15)
#ax2.set_ylim([0,50]) #-10 and 10 
#plt.legend()
plt.setp(ax2.get_xticklabels(), visible=False)
ax2.get_yaxis().set_label_coords(-0.04,0.5)

### Median values before and after perihelion

In [12]:
import math

print(dips3)
print(dips3[7])
'perihelion'
print(dips3[9])

'dips'

beforeph=dips3[0:7]
beforeph = [x for x in beforeph if math.isnan(x) == False]

afterph=dips3[9:len(dips3)]
afterph = [x for x in afterph if math.isnan(x) == False]

print('mean dips value before perihelion',np.mean(beforeph))
print('mean dips value after perihelion',np.mean(afterph))

'peaks'

beforeph2=peaks3[0:7]
beforeph2 = [x for x in beforeph2 if math.isnan(x) == False]

afterph2=peaks3[9:len(peaks3)]
afterph2 = [x for x in afterph2 if math.isnan(x) == False]

print('mean peaks value before perihelion',np.mean(beforeph2))
print('mean peaks value after perihelion',np.mean(afterph2))

'both'

beforeph3=both3[0:7]
beforeph3 = [x for x in beforeph3 if math.isnan(x) == False]

afterph3=both3[9:len(both3)]
afterph3 = [x for x in afterph3 if math.isnan(x) == False]

print('mean both value before perihelion',np.mean(beforeph3))
print('mean both value after perihelion',np.mean(afterph3))

[0.0, 1.7035978282534886e-06, nan, 1.2168729164854033e-05, 1.0133113870646837e-05, 8.209147179063391e-06, 1.4678788270222233e-05, 6.942805171001291e-06, 0.0, 5.23969875225307e-06, 6.9605299283452115e-06, 8.616564889829615e-06, 1.123740407404058e-05, 6.765141352796323e-06, 7.441173149095722e-06, 6.086489008815265e-06]
6.942805171001291e-06
5.23969875225307e-06
mean dips value before perihelion 7.815562718839997e-06
mean dips value after perihelion 7.478143022167969e-06
mean peaks value before perihelion 7.690166274094015e-06
mean peaks value after perihelion 4.49360775541142e-06
mean both value before perihelion 1.0155496744982537e-05
mean both value after perihelion 8.57187310420068e-06


In [13]:
import matplotlib.colors as mcolors
import matplotlib.patches as mpatch
y_errormin=errors2
y_errormax=errors
y_error =[y_errormin, y_errormax]


#FIRST PLOT
plt.figure()
ax7=plt.subplot(111)
figsize=(20,8)
barWidth=0.28
errorcolor='black'
#darkorange darkred forestgreen
plt.bar(x-barWidth,dips3,color='cornflowerblue', width=barWidth, label='Normalised dips')  
plt.errorbar(x-barWidth, dips3, yerr = y_error, fmt ='o', capsize=2,color=errorcolor)
plt.bar(x,peaks3,color='darkorange', width=barWidth, label='Normalised peaks')  
plt.errorbar(x, peaks3, yerr = y_error, fmt ='o', capsize=2,color=errorcolor)
plt.bar(x+barWidth,both3,color='limegreen', width=barWidth, label='Normalised both')  
plt.errorbar(x+barWidth, both3, yerr = y_error, fmt ='o', capsize=2,color=errorcolor)
ax7.set_ylim(0, 3.7e-5)

#plt.xlabel('Days',fontsize=10)
plt.ylabel('# Observations',fontsize=13)
plt.legend()

#x_labels = ['2014-Nov', '2014-Dec','2015-Jan','2015-Feb','2015-Mar','2015-Apr','2015-May','2015-Jun','2015-Jul','2015-Aug','2015-Sep','2015-Oct','2015-Nov','2015-Dec','2016-Jan','2016-Feb'] 
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
#add x-axis values to plot
#plt.xticks(ticks=x_ticks, labels=x_labels,rotation=90,fontsize=8)
plt.xticks(ticks=x, labels=x_labels,fontsize=13)
ax7.get_yaxis().set_label_coords(-0.04,0.5)


## Percent of waves

In [14]:
x=np.arange(0,len(peaks))

plt.figure()
plt.plot(x,peaks2,label='Peaks',color='red')
plt.plot(x,dips2,label='Dips',color='blue')
plt.plot(x,both2,label='Both',color='black')
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
plt.xticks(ticks=x, labels=x_labels,fontsize=13)
plt.ylabel('%', fontsize=13)
plt.ylim([0,100])

#plt
plt.legend()

In [15]:
#FIRST PLOT
plt.figure()
ax7=plt.subplot(313)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.2
plt.bar(x,both2,color='red', width=barWidthnormalized, label='Normalized Data')   
plt.ylabel('% Both', fontsize=13)
ax7.set_ylim([0, 100]) #-5 and 10
#plt.xlabel('Days',fontsize=10)
#plt.legend()

#specify x-axis locations
#x_ticks = [1,31,62,90,121,151,182,212,243,274,304,335,365,396,427,456]

#specify x-axis labels
#x_labels = ['2014-Nov', '2014-Dec','2015-Jan','2015-Feb','2015-Mar','2015-Apr','2015-May','2015-Jun','2015-Jul','2015-Aug','2015-Sep','2015-Oct','2015-Nov','2015-Dec','2016-Jan','2016-Feb'] 
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
#add x-axis values to plot
#plt.xticks(ticks=x_ticks, labels=x_labels,rotation=90,fontsize=8)
plt.xticks(ticks=x, labels=x_labels,fontsize=13)


#SECOND PLOT

ax1=plt.subplot(311,sharex=ax7)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.3
plt.bar(x,peaks2,color='green', width=barWidth, label='Amount of LAP data')    
plt.setp(ax1.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves',fontsize=15)
plt.ylabel('% Peaks', fontsize=13)
#plt.legend()
plt.setp(ax1.get_xticklabels(), visible=False)
ax1.set_ylim([0, 100]) #-5 and 10
#THIRD PLOT

ax2=plt.subplot(312,sharex=ax7)
figsize=(20,4)
barWidth=0.2
barWidthnormalized=0.3
plt.bar(x,dips2, color='blue', width=barWidth, label='Amount of Mirror Mode Waves')    
#plt.setp(ax2.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves 2015',fontsize=15)
#ax2.set_ylim([0,50]) #-10 and 10 
plt.ylabel('% Dips', fontsize=13)
#plt.legend()
plt.setp(ax2.get_xticklabels(), visible=False)
ax2.set_ylim([0, 100]) #-5 and 10

(0.0, 100.0)